# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library, referencing the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) via its Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets and their fields using each entity's `@id`. All subsequent referencing of record sets, fields, and columns will be done via their `@id` following FAIR principles.

The following cell finds all `RecordSet` objects available and lists the fields (by `@id`) for each.

In [ ]:
# Identify all record sets and field @ids

def find_recordsets(m):
    # The Croissant schema typically nests recordSets in metadata.recordSets or uses 'recordSet' property.
    # We'll inspect possible locations for them.
    recordsets = []
    # Try common property names
    if hasattr(m, 'recordSets') and m.recordSets:
        recordsets.extend(m.recordSets)
    if hasattr(m, 'recordSet') and m.recordSet:
        recordsets.extend(m.recordSet)
    # Sometimes present as distributions (tabular data)
    if hasattr(m, 'distribution') and m.distribution:
        for d in m.distribution:
            if hasattr(d, 'recordSets') and d.recordSets:
                recordsets.extend(d.recordSets)
            if hasattr(d, 'recordSet') and d.recordSet:
                recordsets.extend(d.recordSet)
    return recordsets

recordsets = find_recordsets(metadata)
# If none found programmatically, print info and suggest inspecting the ds.records() iterator
if not recordsets:
    print('No explicit record sets found in metadata. Inspecting dataset.records() for available record sets...')
    available_recordsets = dataset.record_sets  # `record_sets` property introduced in mlcroissant>=0.12.1
    for i, rset in enumerate(available_recordsets):
        print(f"[{i}] RecordSet @id: {rset['@id']}")
        if 'fields' in rset and rset['fields']:
            print('    Fields:')
            for field in rset['fields']:
                print(f"        Field @id: {field['@id']}")
    # Save the list of record set @ids for downstream code
    recordset_ids = [r['@id'] for r in available_recordsets]
else:
    for i, rs in enumerate(recordsets):
        print(f"[{i}] RecordSet @id: {getattr(rs, '@id', str(rs))}")
        fields = getattr(rs, 'field', None) or getattr(rs, 'fields', None)
        if fields:
            print('    Fields:')
            for field in fields:
                print(f"        Field @id: {getattr(field, '@id', str(field))}")

## 3. Data Extraction

We now demonstrate extracting data from each available record set. We'll load each set into a pandas DataFrame, referenced by its `@id`. You may substitute `record_set_id`/`field_id` as appropriate based on the output of the previous cell.

In [ ]:
# Extract each record set as a DataFrame. Use @ids from the previous section.

record_sets = recordset_ids  # List of RecordSet @ids from previous cell
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f" --> Loaded {len(records)} records. Columns: {list(dataframes[record_set_id].columns)}\n")

# As an example, show the first record set's columns and sample data
if record_sets:
    example_rs = record_sets[0]
    print(f"Columns for RecordSet {example_rs}:\n{dataframes[example_rs].columns.tolist()}")
    display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)

Let's perform typical EDA operations:
- Select a numeric field (referenced by its field `@id`).
- Filter records with a simple threshold.
- Normalize the numeric field.
- (If available) Group by a chosen categorical field (`@id`).

Update the `numeric_field_id` and `group_field_id` based on your data in the previous section.

In [ ]:
# Example: Adjust these IDs for your dataset
record_set_id = record_sets[0]  # Use first record set
df = dataframes[record_set_id]

# Find a numeric field for analysis (e.g., Age, Diagnosis Interval, etc.)
print('Sample columns:', df.columns.tolist())
# Suppose 'cr:field/age' is the numeric field and 'cr:field/sex' is the group field @id. Adjust as needed.
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
    if 'sex' in col.lower():
        group_field_id = col
if numeric_field_id is None:
    numeric_field_id = df.select_dtypes(include=['number']).columns[0]  # fallback
if group_field_id is None and len(df.select_dtypes(include=['object'])) > 0:
    group_field_id = df.select_dtypes(include=['object']).columns[0]

print(f"Numeric field selected: {numeric_field_id}")
threshold = 50  # Example threshold for age
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} records")
display(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"Grouped mean {numeric_field_id} by {group_field_id} (first 5 groups):")
    display(grouped_df.head())

## 5. Visualization

Visualize distributions and relationships between fields. For example, plot the distribution of the numeric field and compare across categories (group field, if available).
All entity references should use `@id` fields for clarity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id], kde=True, bins=10, color='steelblue')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If group_field_id is available, boxplot by group
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, palette='pastel')
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated loading, inspecting, and performing initial analysis of clinicopathological and molecular data for second primary colorectal cancer in cancer survivors using the [FAIR^2 Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and the `mlcroissant` Python library. All references to data entities throughout the analysis were made via their `@id` fields to ensure reproducibility and FAIR compliance.

You can adapt this notebook further for specific analyses or visualizations as needed for your research or modeling needs.